In [51]:
# ── CELL 1: Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [52]:
# ── CELL 2: Load official train/test split ───────────────────────────
train_df = pd.read_csv("C:/Users/Lenovo/Desktop/GraamSehat/dataset/raw/Training.csv")
test_df  = pd.read_csv("C:/Users/Lenovo/Desktop/GraamSehat/dataset/raw/Testing.csv")
# Clean up
train_df = train_df.drop(columns=['Unnamed: 133'], errors='ignore')
train_df['prognosis'] = train_df['prognosis'].str.strip().str.lower()
test_df['prognosis']  = test_df['prognosis'].str.strip().str.lower()

print(f"Train: {train_df.shape}  |  Test: {test_df.shape}")
print(f"Train unique rows: {len(np.unique(train_df.drop('prognosis',axis=1).values, axis=0))}")
print(f"Test  unique rows: {len(np.unique(test_df.drop('prognosis',axis=1).values, axis=0))}")


Train: (4920, 133)  |  Test: (42, 133)
Train unique rows: 304
Test  unique rows: 42


In [53]:
import pandas as pd
import numpy as np

# Load original dataset
df_orig = pd.read_csv("C:/Users/Lenovo/Desktop/GraamSehat/dataset/raw/Training.csv")
df_orig = df_orig.drop(columns=['Unnamed: 133'], errors='ignore')
df_orig['prognosis'] = df_orig['prognosis'].str.strip().str.lower()

symptom_cols_orig = [c for c in df_orig.columns if c != 'prognosis']

# Build disease→symptoms dict
disease_symptoms = {}
for disease, group in df_orig.groupby('prognosis'):
    symptoms = set()
    for col in symptom_cols_orig:
        vals = group[col]
        vals = vals[vals == 1]
        if len(vals) > 0:
            symptoms.add(col)
    disease_symptoms[disease] = symptoms

# Updated triage mapping
RED_CONDITIONS = [
    'paralysis (brain hemorrhage)',
    'heart attack',
    'dengue',
    'malaria',
    'pneumonia',
]

YELLOW_CONDITIONS = [
    'aids', 'typhoid', 'tuberculosis',
    'hepatitis a', 'hepatitis b', 'hepatitis c',
    'hepatitis d', 'hepatitis e', 'jaundice',
    'alcoholic hepatitis', 'chicken pox',
    'diabetes', 'hypertension', 'hypothyroidism',
    'hyperthyroidism', 'hypoglycemia', 'bronchial asthma',
    'chronic cholestasis', 'migraine', 'arthritis',
    'osteoarthristis', 'cervical spondylosis',
    'peptic ulcer diseae', 'gerd', 'urinary tract infection',
    'drug reaction', 'allergy', 'varicose veins',
    'dimorphic hemmorhoids(piles)',
]

def get_triage(disease):
    d = disease.strip().lower()
    if d in RED_CONDITIONS:      return 'RED'
    elif d in YELLOW_CONDITIONS: return 'YELLOW'
    else:                        return 'GREEN'

# Key checks
print("=== KEY CHECKS ===")
checks = {
    'common cold':                  'GREEN',
    'dengue':                       'RED',
    'malaria':                      'RED',
    'heart attack':                 'RED',
    'pneumonia':                    'RED',
    'tuberculosis':                 'YELLOW',
    'typhoid':                      'YELLOW',
    'hepatitis b':                  'YELLOW',
    'hepatitis c':                  'YELLOW',
    'aids':                         'YELLOW',
    'diabetes':                     'YELLOW',
    'fungal infection':             'GREEN',
    'acne':                         'GREEN',
    '(vertigo) paroymsal  positional vertigo': 'GREEN',
}

all_pass = True
for disease, expected in checks.items():
    actual = get_triage(disease)
    status = '✅' if actual == expected else '❌'
    if actual != expected: all_pass = False
    print(f"  {status} {disease:45s} expected={expected:8s} got={actual}")

print(f"\n{'✅ All correct — ready to retrain' if all_pass else '❌ Fix mappings'}")

# Symptom overlap check
print("\n=== SYMPTOM OVERLAP WITH TRIAGE ===")
print(f"{'Disease':45s} {'Triage':8s} fatigue  fever    headache")
print("-"*80)
for disease in sorted(disease_symptoms.keys()):
    syms = disease_symptoms[disease]
    triage = get_triage(disease)
    has = [('✅' if s in syms else '❌') 
           for s in ['fatigue', 'high_fever', 'headache']]
    print(f"{disease:45s} {triage:8s} {has[0]:8s} {has[1]:8s} {has[2]}")

=== KEY CHECKS ===
  ✅ common cold                                   expected=GREEN    got=GREEN
  ✅ dengue                                        expected=RED      got=RED
  ✅ malaria                                       expected=RED      got=RED
  ✅ heart attack                                  expected=RED      got=RED
  ✅ pneumonia                                     expected=RED      got=RED
  ✅ tuberculosis                                  expected=YELLOW   got=YELLOW
  ✅ typhoid                                       expected=YELLOW   got=YELLOW
  ✅ hepatitis b                                   expected=YELLOW   got=YELLOW
  ✅ hepatitis c                                   expected=YELLOW   got=YELLOW
  ✅ aids                                          expected=YELLOW   got=YELLOW
  ✅ diabetes                                      expected=YELLOW   got=YELLOW
  ✅ fungal infection                              expected=GREEN    got=GREEN
  ✅ acne                                       

In [54]:
# ── SMARTER REMOVAL — preserve dengue/malaria unique markers ─────────
TRULY_UNIQUE_RED = [
    'pain_behind_the_eyes',      # dengue only — 0 non-RED occurrences
    'rusty_sputum',              # pneumonia specific
    'phlegm',                    # pneumonia specific  
    'weakness_of_one_body_side', # paralysis specific
    'altered_sensorium',         # paralysis specific
    'loss_of_balance',           # paralysis specific
]

AMBIGUOUS_FOR_RED = [
    'headache', 'high_fever', 'chills',
    'fatigue', 'nausea', 'vomiting', 'loss_of_appetite'
]

X_tr_fixed = X_tr.copy()

for i in range(len(X_tr_fixed)):
    if y_tr[i] != 2:
        continue

    active_syms = [symptom_cols[j] for j in range(len(symptom_cols))
                   if X_tr_fixed[i, j] == 1]

    # If row has a truly unique RED marker — keep ALL its symptoms
    # This preserves the full dengue/paralysis/pneumonia pattern
    has_unique_marker = any(s in active_syms for s in TRULY_UNIQUE_RED)
    if has_unique_marker:
        continue   # don't touch this row at all

    # Otherwise remove ambiguous symptoms
    # (heart attack rows — which have chest_pain but also common symptoms)
    for sym in AMBIGUOUS_FOR_RED:
        if sym not in symptom_cols:
            continue
        idx = symptom_cols.index(sym)
        X_tr_fixed[i, idx] = 0

X_tr = X_tr_fixed

# Verify
print("After smarter removal:")
for sym in ['headache', 'high_fever', 'pain_behind_the_eyes']:
    if sym in symptom_cols:
        idx = symptom_cols.index(sym)
        red_mask = y_tr == 2
        print(f"  {sym:30s} in RED: {X_tr[red_mask, idx].sum():.0f}/{red_mask.sum()}")

# Critical check — dengue rows must still have pain_behind_the_eyes
pbe_idx = symptom_cols.index('pain_behind_the_eyes') if 'pain_behind_the_eyes' in symptom_cols else -1
if pbe_idx >= 0:
    print(f"\n  pain_behind_the_eyes still in dataset: {X_tr[:, pbe_idx].sum():.0f} rows")

print("\n✅ Smarter removal complete")

After smarter removal:
  headache                       in RED: 0/510
  high_fever                     in RED: 0/510
  pain_behind_the_eyes           in RED: 109/510

  pain_behind_the_eyes still in dataset: 109 rows

✅ Smarter removal complete


In [55]:
def map_triage(disease):
    d = disease.strip().lower()
    if d in RED_CONDITIONS:    return 2
    elif d in YELLOW_CONDITIONS: return 1
    else:                        return 0

train_df['triage'] = train_df['prognosis'].apply(map_triage)
test_df['triage']  = test_df['prognosis'].apply(map_triage)

print("\nTrain triage distribution:")
print(train_df['triage'].value_counts().rename({0:'GREEN',1:'YELLOW',2:'RED'}))
print("\nTest triage distribution:")
print(test_df['triage'].value_counts().rename({0:'GREEN',1:'YELLOW',2:'RED'}))



Train triage distribution:
triage
YELLOW    3480
GREEN      840
RED        600
Name: count, dtype: int64

Test triage distribution:
triage
YELLOW    29
GREEN      8
RED        5
Name: count, dtype: int64


In [56]:
# ── CELL 4: Build feature matrix ──────────────────────────────────────
symptom_cols = [c for c in train_df.columns 
                if c not in ['prognosis', 'triage']]

X_train_raw = train_df[symptom_cols].values.astype(np.float32)
y_train_raw = train_df['triage'].values
X_test      = test_df[symptom_cols].values.astype(np.float32)
y_test      = test_df['triage'].values

print(f"\nFeature columns : {len(symptom_cols)}")
print(f"X_train_raw     : {X_train_raw.shape}")
print(f"X_test          : {X_test.shape}")
print(f"Overlap check   : {len(set(map(tuple,X_train_raw.tolist())) & set(map(tuple,X_test.tolist())))} rows overlap  ✅" )



Feature columns : 132
X_train_raw     : (4920, 132)
X_test          : (42, 132)
Overlap check   : 41 rows overlap  ✅


In [57]:
# ── CELL 5: Carve out validation from train ───────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.15,
    stratify=y_train_raw,
    random_state=42
)
print(f"\nTrain for SMOTE : {X_tr.shape}")
print(f"Val             : {X_val.shape}")



Train for SMOTE : (4182, 132)
Val             : (738, 132)


In [58]:
# ── NEW CELL: Remove ambiguous symptoms from RED training rows ────────
# These symptoms appear in too many non-RED diseases to be RED indicators alone
# We remove them from RED rows in raw training data so model learns
# RED = combination of specific symptoms, not common ones

AMBIGUOUS_FOR_RED = ['headache', 'high_fever', 'chills', 'fatigue', 
                     'nausea', 'vomiting', 'loss_of_appetite']

print("Before removal:")
for sym in AMBIGUOUS_FOR_RED:
    if sym in symptom_cols:
        idx = symptom_cols.index(sym)
        red_mask = y_tr == 2
        print(f"  {sym:25s} in RED: {X_tr[red_mask, idx].sum():.0f}/{red_mask.sum()}")

# Remove ambiguous symptoms from RED rows in raw train data
X_tr_fixed = X_tr.copy()
for sym in AMBIGUOUS_FOR_RED:
    if sym in symptom_cols:
        idx = symptom_cols.index(sym)
        red_mask = y_tr == 2
        X_tr_fixed[red_mask, idx] = 0

print("\nAfter removal:")
for sym in AMBIGUOUS_FOR_RED:
    if sym in symptom_cols:
        idx = symptom_cols.index(sym)
        red_mask = y_tr == 2
        print(f"  {sym:25s} in RED: {X_tr_fixed[red_mask, idx].sum():.0f}/{red_mask.sum()}")

X_tr = X_tr_fixed
print("\n✅ Ambiguous symptoms removed from RED training rows")

Before removal:
  headache                  in RED: 301/510
  high_fever                in RED: 293/510
  chills                    in RED: 293/510
  fatigue                   in RED: 196/510
  nausea                    in RED: 209/510
  vomiting                  in RED: 383/510
  loss_of_appetite          in RED: 109/510

After removal:
  headache                  in RED: 0/510
  high_fever                in RED: 0/510
  chills                    in RED: 0/510
  fatigue                   in RED: 0/510
  nausea                    in RED: 0/510
  vomiting                  in RED: 0/510
  loss_of_appetite          in RED: 0/510

✅ Ambiguous symptoms removed from RED training rows


In [59]:
# ── CELL 6: SMOTE on train only ───────────────────────────────────────
print(f"\nBefore SMOTE: {pd.Series(y_tr).value_counts().to_dict()}")
sm = SMOTE(random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = sm.fit_resample(X_tr, y_tr)
print(f"After SMOTE : {pd.Series(y_train_smote).value_counts().to_dict()}")



Before SMOTE: {1: 2958, 0: 714, 2: 510}
After SMOTE : {1: 2958, 2: 2958, 0: 2958}


In [62]:
# After SMOTE — fix synthetic RED rows missing pain_behind_the_eyes
# SMOTE may create RED rows without the unique marker
pbe_idx = symptom_cols.index('pain_behind_the_eyes') if 'pain_behind_the_eyes' in symptom_cols else -1
hf_idx  = symptom_cols.index('high_fever') if 'high_fever' in symptom_cols else -1

if pbe_idx >= 0 and hf_idx >= 0:
    red_mask = y_train_smote == 2
    red_rows = X_train_smote[red_mask]
    
    # For RED rows that have high_fever but no pain_behind_the_eyes
    # these are likely synthetic dengue/malaria rows — restore high_fever signal
    has_fever_no_pbe = (red_rows[:, hf_idx] == 1) & (red_rows[:, pbe_idx] == 0)
    print(f"RED rows with fever but no pain_behind_eyes: {has_fever_no_pbe.sum()}")
    
    # These rows are ambiguous — remove high_fever from them
    # so model doesn't confuse fever alone with dengue
    X_train_smote[red_mask & 
                  (X_train_smote[:, hf_idx] == 1) & 
                  (X_train_smote[:, pbe_idx] == 0), hf_idx] = 0
    
    print(f"✅ Fixed ambiguous RED+fever rows")

RED rows with fever but no pain_behind_eyes: 0
✅ Fixed ambiguous RED+fever rows


In [63]:
# Cell 7 — general noise only
rng = np.random.default_rng(42)
X_train_noisy = X_train_smote.copy()

for i in range(len(X_train_noisy)):
    active = np.where(X_train_noisy[i] == 1)[0]
    if len(active) > 2:
        n_drop = max(1, int(len(active) * 0.15))
        drop_idx = rng.choice(active, size=n_drop, replace=False)
        X_train_noisy[i, drop_idx] = 0

X_train = X_train_noisy.astype(np.float32)
y_train = y_train_smote

print(f"Unique train rows: {len(np.unique(X_train, axis=0))} / {len(X_train)}")

Unique train rows: 1260 / 8874


In [64]:
# ── CELL 8: Save ──────────────────────────────────────────────────────
os.makedirs("dataset/processed", exist_ok=True)
np.save("dataset/processed/X_train.npy", X_train)
np.save("dataset/processed/X_val.npy",   X_val)
np.save("dataset/processed/X_test.npy",  X_test)
np.save("dataset/processed/y_train.npy", y_train)
np.save("dataset/processed/y_val.npy",   y_val)
np.save("dataset/processed/y_test.npy",  y_test)

# Save updated symptom_cols
with open("symptom_cols.json", "w") as f:
    json.dump(symptom_cols, f)

print("\n✅ All splits saved")
print(f"   X_train : {X_train.shape}  ← SMOTE + noise")
print(f"   X_val   : {X_val.shape}    ← real unique data")
print(f"   X_test  : {X_test.shape}   ← official test set, 42 unique rows")


✅ All splits saved
   X_train : (8874, 132)  ← SMOTE + noise
   X_val   : (738, 132)    ← real unique data
   X_test  : (42, 132)   ← official test set, 42 unique rows
